# Preprocessing & Kiểm Tra Ảnh (`inspect_and_run.ipynb`)

0. Tải data trước (1 lần): `python src/download_data.py` → ảnh nằm ở `data/raw/{Drowsy,Non Drowsy}`.
1. Nạp cấu hình `configs/preprocess.yaml`.
2. Chạy `build_video_and_frame_manifests` để sinh `data/processed/labels.csv` và `data/processed/frames_manifest.csv` (giữ nguyên cột `split`).
3. Chọn `VIDEO_ID` hoặc `IMAGE_PATH` bằng biến Python để xem ảnh và chạy thử YOLOv8.

In [ ]:
import sys
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

def find_repo_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "configs" / "preprocess.yaml").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy configs/preprocess.yaml")

ROOT = find_repo_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from preprocessing.build_labels import build_video_and_frame_manifests, load_preprocess_config

cfg = load_preprocess_config(ROOT / "configs" / "preprocess.yaml")

# Tùy chọn: trỏ sang thư mục khác thay vì data/raw trong config (hoặc đặt biến môi trường DDD_RAW_DIR)
RAW_DATA_DIR = None  # ví dụ: Path(r"D:/datasets/drowsiness_raw")
if RAW_DATA_DIR is not None:
    cfg = replace(cfg, raw_data_dir=Path(RAW_DATA_DIR))

print("Raw data dir:", cfg.raw_data_dir if cfg.raw_data_dir.is_absolute() else ROOT / cfg.raw_data_dir)

In [ ]:
summaries, records = build_video_and_frame_manifests(cfg, base_dir=ROOT)

labels_path = cfg.output_labels_csv if cfg.output_labels_csv.is_absolute() else ROOT / cfg.output_labels_csv
frames_path = cfg.output_frames_manifest_csv if cfg.output_frames_manifest_csv.is_absolute() else ROOT / cfg.output_frames_manifest_csv

df_labels = pd.read_csv(labels_path)
df_frames = pd.read_csv(frames_path)

print(f"Videos: {len(df_labels)} | Frames: {len(df_frames)}")
df_labels

In [ ]:
# Chọn video_id cần xem (None = tự lấy video đầu tiên)
VIDEO_ID = None
START_POS = 0
MAX_SHOW = 8

if df_labels.empty:
    print("Chưa có ảnh trong thư mục dữ liệu.")
else:
    vid = VIDEO_ID or df_labels["video_id"].iloc[0]
    window = df_frames[df_frames["video_id"] == vid].sort_values("frame_idx").iloc[START_POS : START_POS + MAX_SHOW]

    cols = min(4, len(window))
    rows = (len(window) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows))
    axes_list = [axes] if rows * cols == 1 else list(axes.flat)
    for ax in axes_list:
        ax.axis("off")

    for ax, (_, row) in zip(axes_list, window.iterrows()):
        img_path = Path(row["image_path"])
        if not img_path.is_absolute():
            img_path = ROOT / img_path
        with Image.open(img_path) as im:
            ax.imshow(im.convert("RGB"))
        ax.set_title(f"{row['video_id']}_{row['frame_idx']} (label={row['label']})")

    plt.tight_layout()
    plt.show()

In [ ]:
# Chọn 1 ảnh chạy thử YOLOv8 (None = lấy ảnh đầu tiên trong df_frames)
IMAGE_PATH = None  # ví dụ: ROOT / "data/raw/Drowsy/A0001.png"
YOLO_WEIGHTS = "yolov8n.pt"
CONF_THRESH = 0.25

target_img = IMAGE_PATH
if target_img is None and not df_frames.empty:
    first = Path(df_frames["image_path"].iloc[0])
    target_img = first if first.is_absolute() else ROOT / first

if target_img is None:
    print("Chưa có ảnh để chạy thử YOLO.")
else:
    import cv2
    from ultralytics import YOLO

    model = YOLO(YOLO_WEIGHTS)
    res = model.predict(source=str(target_img), conf=CONF_THRESH, verbose=False)
    rgb = cv2.cvtColor(res[0].plot(), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(6, 4))
    plt.imshow(rgb)
    plt.title(Path(target_img).name)
    plt.axis("off")
    plt.show()